# 01 — Data Preparation

**From Clinical Case Reports to Knowledge Graphs**

This notebook is Step 0 of the activity plan: it downloads the source files
from the [MultiCaRe dataset on Zenodo](https://doi.org/10.5281/zenodo.10079369),
loads them into a DuckDB database, and exports the same tables as CSV — a
`full` copy and a fixed 50-article `sample` copy. Later notebooks connect to
one of these three forms without re-downloading or re-parsing anything:

- [`02a_csv_data_exploraton_sample.ipynb`](02a_csv_data_exploraton_sample.ipynb) — explores the 50-article CSV sample
- [`02b_csv_data_exploraton_full.ipynb`](02b_csv_data_exploraton_full.ipynb) — explores the full CSV export
- [`03_duckdb_data_exploraton.ipynb`](03_duckdb_data_exploraton.ipynb) — explores the DuckDB database directly

Only the files actually used by the activity are downloaded (`cases.parquet`,
`metadata.parquet`, `data_dictionary.csv`). The dataset also ships PubMed
Central image archives and image-caption tables (several GB) that this
activity does not use, so they are skipped.


## 1. Setup

Directory layout produced by this notebook (created automatically if missing;
everything under `data/` is gitignored):

```
to-kg/
├── data/
│   ├── raw/            <- downloaded Zenodo files
│   ├── duckdb/          <- clinical_cases.duckdb
│   └── csv/
│       ├── full/        <- CSV export of every table, all rows
│       └── sample/      <- CSV export, fixed 50-article random sample
└── notebooks/
    └── 01_data_preparation.ipynb   <- this notebook
```


In [1]:
import random
from pathlib import Path

import duckdb
import requests
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
DUCKDB_DIR = DATA_DIR / "duckdb"
CSV_FULL_DIR = DATA_DIR / "csv" / "full"
CSV_SAMPLE_DIR = DATA_DIR / "csv" / "sample"

for d in (RAW_DIR, DUCKDB_DIR, CSV_FULL_DIR, CSV_SAMPLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DUCKDB_DIR / "clinical_cases.duckdb"

SAMPLE_SIZE = 50
SAMPLE_SEED = 42  # fixed so the CSV sample is identical across runs

DATA_DIR


PosixPath('/home/santanche/git/2learn/nlp2learn/to-kg/data')

## 2. Download source files from Zenodo

We resolve the **concept DOI** (`10.5281/zenodo.10079369`) through Zenodo's
`versions/latest` endpoint, so this notebook always fetches the current
version of the dataset instead of a version hardcoded at the time this
notebook was written. Files already present with the correct size are not
re-downloaded, so re-running this cell is cheap.


In [2]:
ZENODO_CONCEPT_ID = "10079369"  # concept DOI 10.5281/zenodo.10079369
FILES_TO_DOWNLOAD = ["cases.parquet", "metadata.parquet", "data_dictionary.csv"]


def get_latest_record(concept_id: str) -> dict:
    url = f"https://zenodo.org/api/records/{concept_id}/versions/latest"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def download_file(url: str, destination: Path, expected_size: int) -> None:
    if destination.exists() and destination.stat().st_size == expected_size:
        print(f"  {destination.name} already present ({expected_size:,} bytes), skipping")
        return
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(destination, "wb") as fh, tqdm(
            total=expected_size, unit="B", unit_scale=True, desc=destination.name
        ) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                fh.write(chunk)
                bar.update(len(chunk))


In [3]:
record = get_latest_record(ZENODO_CONCEPT_ID)
print(
    f"Latest version: {record['metadata']['version']} "
    f"(record id {record['id']}, published {record['metadata']['publication_date']})"
)

files_by_key = {f["key"]: f for f in record["files"]}

for key in FILES_TO_DOWNLOAD:
    file_info = files_by_key[key]
    download_file(file_info["links"]["self"], RAW_DIR / key, file_info["size"])


Latest version: 3.0.1 (record id 20416562, published 2026-05-27)
  cases.parquet already present (168,061,894 bytes), skipping
  metadata.parquet already present (20,357,281 bytes), skipping
  data_dictionary.csv already present (6,291 bytes), skipping


## 3. Build the DuckDB database

Building an explicit DuckDB database file is not required for the
information-extraction steps later in the activity — DuckDB can query
Parquet files directly. We build it anyway because it gives students a
single, fast, queryable artifact (`data/duckdb/clinical_cases.duckdb`) to
explore the corpus before any NLP work starts.

**Note on the raw file layout:** both source files are more nested than
`data_dictionary.csv` documents. `cases.parquet` has one row per *article*,
with a `cases` column holding a list of structs (one struct per patient
case: `age`, `case_id`, `case_text`, `gender`). `metadata.parquet` has one
row per article too, with every documented field (`title`, `authors`,
`journal`, `year`, `doi`, `pmid`, `mesh_terms`, `case_amount`, …) packed
into a single `article_metadata` struct column. The `CREATE TABLE`
statements below unnest/flatten both so the resulting `cases` and
`metadata` tables match the flat, one-row-per-case / one-row-per-article
columns the data dictionary describes.


In [4]:
con = duckdb.connect(str(DB_PATH))

con.execute(f"""
    CREATE OR REPLACE TABLE cases AS
    SELECT article_id, case_entry.*
    FROM (
        SELECT article_id, UNNEST(cases) AS case_entry
        FROM read_parquet('{(RAW_DIR / "cases.parquet").as_posix()}')
    )
""")

con.execute(f"""
    CREATE OR REPLACE TABLE metadata AS
    SELECT article_id, article_metadata.*
    FROM read_parquet('{(RAW_DIR / "metadata.parquet").as_posix()}')
""")

con.execute(f"""
    CREATE OR REPLACE TABLE data_dictionary AS
    SELECT * FROM read_csv_auto('{(RAW_DIR / "data_dictionary.csv").as_posix()}')
""")

con.sql("SHOW TABLES")


┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ cases           │
│ data_dictionary │
│ metadata        │
└─────────────────┘

In [5]:
con.sql("""
    SELECT 'cases' AS table_name, COUNT(*) AS n_rows FROM cases
    UNION ALL
    SELECT 'metadata', COUNT(*) FROM metadata
    UNION ALL
    SELECT 'data_dictionary', COUNT(*) FROM data_dictionary
""")


┌─────────────────┬────────┐
│   table_name    │ n_rows │
│     varchar     │ int64  │
├─────────────────┼────────┤
│ cases           │  98641 │
│ metadata        │  76137 │
│ data_dictionary │     45 │
└─────────────────┴────────┘

## 4. Confronting the extracted metadata with the source paper

The dataset description paper (Nievas Offidani et al., *An Open-Source
Clinical Case Dataset for Medical Image Classification and Multimodal AI
Applications*, *Data*, 2026, [doi:10.3390/data10080123](https://doi.org/10.3390/data10080123))
reports approximately **93,816 clinical cases** in the abstract. Because this
notebook always downloads the *latest* Zenodo version (Section 2), while the
paper describes a snapshot taken at publication time, some drift between the
two counts is expected — the query below quantifies it rather than assuming
the numbers must match.


In [6]:
PAPER_REPORTED_CASES = 93_816  # abstract of Nievas Offidani et al., Data (2026)

actual_cases = con.sql("SELECT COUNT(*) FROM cases").fetchone()[0]
diff = actual_cases - PAPER_REPORTED_CASES

print(f"Cases in the downloaded dataset version {record['metadata']['version']}: {actual_cases:,}")
print(f"Cases reported in the paper:                                    {PAPER_REPORTED_CASES:,}")
print(f"Difference:                                                     {diff:+,} ({diff / PAPER_REPORTED_CASES:+.1%})")


Cases in the downloaded dataset version 3.0.1: 98,641
Cases reported in the paper:                                    93,816
Difference:                                                     +4,825 (+5.1%)


## 5. Export the full tables to CSV

`data/csv/full/` gets one CSV per DuckDB table, all rows. `authors`,
`mesh_terms`, `major_mesh_terms`, and `keywords` are `VARCHAR[]` in DuckDB;
CSV has no array type, so DuckDB writes them as bracketed text
(e.g. `[Female]`) and reading the CSV back infers plain `VARCHAR` — a schema
drift comparable to the Parquet-vs-`data_dictionary.csv` drift described in
[`docs/data_source.md`](../docs/data_source.md). `year` shows the same
effect the other way: it is `VARCHAR` in the DuckDB table but round-trips
through CSV as `BIGINT`, since every value happens to look numeric.


In [7]:
for table in ("cases", "metadata", "data_dictionary"):
    dest = CSV_FULL_DIR / f"{table}.csv"
    con.execute(f"COPY {table} TO '{dest.as_posix()}' (HEADER, DELIMITER ',')")
    print(f"  {dest.relative_to(PROJECT_DIR)}")


  data/csv/full/cases.csv
  data/csv/full/metadata.csv
  data/csv/full/data_dictionary.csv


## 6. Export a fixed 50-article sample to CSV

`data/csv/sample/` is meant for quick, low-friction exploration (small
enough to open in a text editor or spreadsheet) without losing the
`cases` ↔ `metadata` relationship. We sample **articles**, not cases or
metadata rows independently: 50 `article_id`s are drawn with a fixed seed
from the full, sorted list, then both `cases` and `metadata` are filtered to
that same set of articles. `metadata` ends up with exactly 50 rows
(`article_id` is its primary key); `cases` ends up with slightly more, since
some articles contribute more than one case. `data_dictionary` documents
field meaning rather than case/article data, so it is copied over
unfiltered, same as in the full export.

The seed is fixed (`SAMPLE_SEED = 42`) so re-running this notebook always
reproduces the same 50 articles, as long as the set of `article_id`s in the
downloaded dataset hasn't changed.


In [8]:
article_ids = [
    row[0] for row in con.sql("SELECT article_id FROM metadata ORDER BY article_id").fetchall()
]
sample_article_ids = sorted(random.Random(SAMPLE_SEED).sample(article_ids, SAMPLE_SIZE))

print(f"Sampled {len(sample_article_ids)} of {len(article_ids)} articles (seed={SAMPLE_SEED})")
sample_article_ids[:5]


Sampled 50 of 76137 articles (seed=42)


['PMC10106591', 'PMC10405809', 'PMC10434843', 'PMC10488652', 'PMC10521634']

In [9]:
con.execute("CREATE OR REPLACE TEMP TABLE sample_article_ids (article_id VARCHAR)")
con.executemany("INSERT INTO sample_article_ids VALUES (?)", [(a,) for a in sample_article_ids])

con.sql("""
    SELECT 'cases' AS table_name, COUNT(*) AS n_rows
    FROM cases WHERE article_id IN (SELECT article_id FROM sample_article_ids)
    UNION ALL
    SELECT 'metadata', COUNT(*)
    FROM metadata WHERE article_id IN (SELECT article_id FROM sample_article_ids)
""")


┌────────────┬────────┐
│ table_name │ n_rows │
│  varchar   │ int64  │
├────────────┼────────┤
│ cases      │     56 │
│ metadata   │     50 │
└────────────┴────────┘

In [10]:
con.execute(f"""
    COPY (
        SELECT * FROM cases WHERE article_id IN (SELECT article_id FROM sample_article_ids)
    ) TO '{(CSV_SAMPLE_DIR / "cases.csv").as_posix()}' (HEADER, DELIMITER ',')
""")
con.execute(f"""
    COPY (
        SELECT * FROM metadata WHERE article_id IN (SELECT article_id FROM sample_article_ids)
    ) TO '{(CSV_SAMPLE_DIR / "metadata.csv").as_posix()}' (HEADER, DELIMITER ',')
""")
con.execute(f"""
    COPY data_dictionary TO '{(CSV_SAMPLE_DIR / "data_dictionary.csv").as_posix()}' (HEADER, DELIMITER ',')
""")

for dest in (CSV_SAMPLE_DIR / "cases.csv", CSV_SAMPLE_DIR / "metadata.csv", CSV_SAMPLE_DIR / "data_dictionary.csv"):
    print(f"  {dest.relative_to(PROJECT_DIR)}")


  data/csv/sample/cases.csv
  data/csv/sample/metadata.csv
  data/csv/sample/data_dictionary.csv


## 7. Wrap up

Three artifacts now exist under `data/`, all gitignored and all built from
the same downloaded Parquet/CSV files in `data/raw/`:

- `data/duckdb/clinical_cases.duckdb` — `cases`, `metadata`, `data_dictionary`
  tables, explored in [`03_duckdb_data_exploraton.ipynb`](03_duckdb_data_exploraton.ipynb)
- `data/csv/full/*.csv` — the same three tables, all rows, explored in
  [`02b_csv_data_exploraton_full.ipynb`](02b_csv_data_exploraton_full.ipynb)
- `data/csv/sample/*.csv` — the same three tables restricted to a fixed
  50-article sample, explored in
  [`02a_csv_data_exploraton_sample.ipynb`](02a_csv_data_exploraton_sample.ipynb)

We close the connection here so the DuckDB file is not left locked.


In [11]:
con.close()
